# Pairwise TF-IDF Knowledge Extraction Experiments
This notebook demonstrates and evaluates the pairwise TF-IDF method against standard baseline keyword extractors (Vanilla TF-IDF and RAKE).

## 1. Imports and Setup

In [ ]:
import sys
import os

# Add src directory to path
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import preprocess_text, remove_stopwords, pair_book_and_author
from vanilla_tfidf import VanillaTFIDFExtractor
from rake_extractor import RakeExtractor
from pairwise_tfidf import PairwiseTFIDFExtractor

## 2. Generate/Load Dummy Dataset
For demonstration purposes, let's create a small dummy dataset pairing book synopses with author biographies.

In [ ]:
# Mock data containing author profiles and their books
books_data = pd.DataFrame({
    'author_id': [1, 2, 3],
    'book_title': ['The Stars Beyond', 'Python Magic', 'Deep Learning Quest'],
    'synopsis': [
        'A hard science fiction odyssey exploring interstellar space travel, galaxies, stars, and alien civilization.',
        'An advanced coding guide to programming structures, magic methods, decorators, and optimization in Python.',
        'A comprehensive guide on neural networks, backpropagation, transformers, deep learning, and computer vision.'
    ]
})

authors_data = pd.DataFrame({
    'author_id': [1, 2, 3],
    'author_name': ['Dr. Stella Astro', 'Guido PyCoder', 'Prof. Neural Deep'],
    'bio': [
        'Stella Astro is an astrophysicist and science fiction writer specializing in galaxy structures, space flight, and interstellar physics.',
        'Guido PyCoder is a veteran software engineer who loves coding python scripts, software architecture, and optimization tools.',
        'Prof. Neural Deep researches neural networks, deep learning algorithms, computer vision models, and artificial intelligence.'
    ]
})

paired_df = pair_book_and_author(books_data, authors_data)
paired_df

## 3. Preprocessing

In [ ]:
# Apply preprocessing
paired_df['clean_synopsis'] = paired_df['synopsis'].apply(preprocess_text).apply(remove_stopwords)
paired_df['clean_bio'] = paired_df['bio'].apply(preprocess_text).apply(remove_stopwords)
paired_df

## 4. Run Baselines

### 4.1 Vanilla TF-IDF Extraction (on Book Synopsis)

In [ ]:
vanilla_extractor = VanillaTFIDFExtractor()
# Fit on synopses
vanilla_extractor.fit_transform(paired_df['clean_synopsis'].tolist())

print("Vanilla TF-IDF top keywords for the first book:")
print(vanilla_extractor.extract_top_keywords(doc_idx=0, top_n=5))

### 4.2 RAKE Baseline (on Book Synopsis)

In [ ]:
rake_extractor = RakeExtractor()
print("RAKE top keywords for the first book:")
print(rake_extractor.extract_keywords(paired_df['synopsis'].iloc[0], top_n=5))

## 5. Run Pairwise TF-IDF Extraction

In [ ]:
pairwise_extractor = PairwiseTFIDFExtractor()

# Fit vocabulary/IDF on the combined documents (synopses + bios)
all_docs = paired_df['clean_synopsis'].tolist() + paired_df['clean_bio'].tolist()
pairwise_extractor.fit(all_docs)

print("Pairwise TF-IDF mutual keywords for Book-Author pair 1 (Stella Astro):")
for metric in ['product', 'min', 'harmonic']:
    keywords = pairwise_extractor.extract_pairwise_keywords(
        doc_a=paired_df['clean_synopsis'].iloc[0],
        doc_b=paired_df['clean_bio'].iloc[0],
        metric=metric,
        top_n=5
    )
    print(f"Metric '{metric}':", keywords)

## 6. Comparison & Visualization

In [ ]:
# Visualizing comparison for stella astro's book/bio pair
vanilla_res = dict(vanilla_extractor.extract_top_keywords(doc_idx=0, top_n=5))
rake_res = dict(rake_extractor.extract_keywords(paired_df['synopsis'].iloc[0], top_n=5))
pairwise_res = dict(pairwise_extractor.extract_pairwise_keywords(
    doc_a=paired_df['clean_synopsis'].iloc[0],
    doc_b=paired_df['clean_bio'].iloc[0],
    metric='product',
    top_n=5
))

print("Vanilla:", vanilla_res)
print("RAKE:", rake_res)
print("Pairwise (Product):", pairwise_res)